# Blue vs Red: Investigating Side Advantage in Professional League of Legends

**Weder Qin, Kevin Cui**


**Original project website:**  
https://weder1126.github.io/lol-blue-vs-red/

## Setup

Download the 2022 Oracle's Elixir professional League of Legends dataset and place it at:

`data/2022_LoL_esports_match_data_from_OraclesElixir.csv`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
import plotly.io as pio

pd.options.plotting.backend = 'plotly'
pd.set_option('display.max_columns', 60)
np.random.seed(42)
pio.renderers.default = "plotly_mimetype+png"

## Step 1: Introduction

Competitive *League of Legends* is one of the largest and longest-running e-sports in
the world. Professional teams are assigned either the **Blue** or **Red** side of the
map, and players and analysts have long debated whether map side provides a meaningful
competitive advantage.

### Research question

> **In professional League of Legends matches from the 2022 season, is the win rate of
> teams on Blue side different from teams on Red side?**

The project uses the 2022 Oracle's Elixir dataset. The submitted analysis reported
**150,348 rows and 165 columns** in the raw data. Each game contributes player rows
plus two team-summary rows; restricting to `position == 'team'` gives **25,058
team-game observations across 12,529 unique games**.

Important columns include `gameid`, `league`, `date`, `side`, `position`, `teamname`,
`result`, `gamelength`, `kills`, `deaths`, `assists`, `golddiffat15`, `patch`, and
`playoffs`.

In [ ]:
# Load the raw 2022 Oracle's Elixir CSV
data_path = Path("data") / "2022_LoL_esports_match_data_from_OraclesElixir.csv"

raw = pd.read_csv(data_path, low_memory=False)
print("Raw shape:", raw.shape)
raw.head(3)

## Step 2: Data Cleaning and Exploratory Data Analysis

The submitted project:

1. Restricts the data to team-summary rows.
2. Parses `date` as datetime.
3. Drops rows missing `side` or `result`.
4. Engineers `win`, `game_minutes`, and `kda` for exploratory analysis.
5. Uses a selected set of tier-1 leagues for league-faceted visualizations while
   retaining the full dataset for the hypothesis test.

Post-game quantities such as KDA and game length are used for EDA only and are
deliberately excluded from predictive modeling to avoid outcome leakage.

In [ ]:
# --- Cleaning pipeline ---

def keep_team_rows(df):
    return df.loc[df['position'] == 'team'].copy()

def parse_date(df):
    return df.assign(date=pd.to_datetime(df['date'], errors='coerce'))

def drop_missing_outcome(df):
    return df.dropna(subset=['side', 'result'])

def add_features(df):
    # New columns engineered from existing ones (EDA only -- not model features).
    return df.assign(
        win=df['result'].astype(int),
        game_minutes=df['gamelength'] / 60,
        kda=(df['kills'] + df['assists']) / df['deaths'].replace(0, 1),
    )

team = (
    raw
    .pipe(keep_team_rows)
    .pipe(parse_date)
    .pipe(drop_missing_outcome)
    .pipe(add_features)
    .reset_index(drop=True)
)

print('Team-rows shape after cleaning:', team.shape)
print('Unique games:', team['gameid'].nunique())
print('Side distribution:', team['side'].value_counts().to_dict())
print(f"Overall Blue win rate: {team.loc[team['side'] == 'Blue', 'result'].mean():.4f}")
print(f"Overall Red win rate: {team.loc[team['side'] == 'Red', 'result'].mean():.4f}")

team[['gameid', 'league', 'date', 'side', 'result', 'game_minutes', 'kda']].head()

The original submitted run reported:

- Team rows after cleaning: **25,058**
- Unique games: **12,529**
- Blue rows: **12,529**
- Red rows: **12,529**
- Overall Blue win rate: **0.5248**
- Overall Red win rate: **0.4752**

In [ ]:
# --- Headline bivariate visualization: win rate by map side and league ---

TIER1 = ['LCK', 'LPL', 'LEC', 'LCS', 'CBLOL', 'LJL', 'PCS', 'VCS', 'WLDs', 'MSI']

tier1 = team.loc[team['league'].isin(TIER1)]

winrate = (
    tier1
    .groupby(['league', 'side'], as_index=False)['result']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'win_rate', 'count': 'games'})
)

fig = px.bar(
    winrate,
    x='league',
    y='win_rate',
    color='side',
    barmode='group',
    color_discrete_map={'Blue': '#1f77b4', 'Red': '#d62728'},
    category_orders={'league': TIER1, 'side': ['Blue', 'Red']},
    text=winrate['win_rate'].apply(lambda x: f'{x:.1%}'),
    hover_data={'games': True, 'win_rate': ':.3f'},
    title='Blue- vs Red-side win rate per tier-1 league (2022 season)',
    labels={'win_rate': 'Win rate', 'league': 'League', 'side': 'Map side'},
)

fig.update_layout(
    yaxis_tickformat='.0%',
    yaxis_range=[0.35, 0.65],
    bargap=0.25,
    legend_title_text='Map side',
    template='plotly_white',
    height=480,
    width=900,
)

fig.add_hline(
    y=0.5,
    line_dash='dash',
    line_color='gray',
    annotation_text='50% — no side advantage',
    annotation_position='top left',
)

fig.update_traces(textposition='outside')
fig.show()

In [ ]:
# Quick aggregate table backing up the plot
(
    tier1
    .groupby(['league', 'side'])
    .agg(games=('result', 'count'), win_rate=('result', 'mean'))
    .round(3)
)

### Additional univariate and bivariate analysis

The submitted analysis also examined:

- the distribution of game length;
- the number of team-games per league;
- game length versus outcome;
- the Blue-minus-Red win-rate gap by league.

The report noted that most games last roughly **25–35 minutes**, that league coverage is
uneven, and that Blue-side advantage varies substantially by league.

In [ ]:
# --- Univariate analysis ---

fig_len = px.histogram(
    team,
    x='game_minutes',
    nbins=50,
    title='Univariate: distribution of game length (minutes)',
    labels={'game_minutes': 'Game length (minutes)'},
)
fig_len.update_layout(template='plotly_white', height=380, width=750)
fig_len.show()

games_per_league = (
    team['league']
    .value_counts()
    .head(15)
    .rename_axis('league')
    .reset_index(name='games')
)

fig_lg = px.bar(
    games_per_league,
    x='league',
    y='games',
    title='Univariate: number of team-games per league (top 15)',
    labels={'league': 'League', 'games': 'Team-games'},
)
fig_lg.update_layout(template='plotly_white', height=380, width=750)
fig_lg.show()

# --- Bivariate analysis: game length by outcome ---

fig_box = px.box(
    team.assign(outcome=team['result'].map({1: 'Win', 0: 'Loss'})),
    x='outcome',
    y='game_minutes',
    color='outcome',
    title='Bivariate: game length by outcome',
    labels={'outcome': 'Outcome', 'game_minutes': 'Game length (minutes)'},
)
fig_box.update_layout(template='plotly_white', height=400, width=750, showlegend=False)
fig_box.show()

In [ ]:
# Interesting aggregate: Blue-minus-Red win-rate gap by league

agg = (
    tier1
    .pivot_table(index='league', columns='side', values='win', aggfunc='mean')
    .assign(blue_minus_red=lambda d: d['Blue'] - d['Red'])
    .sort_values('blue_minus_red', ascending=False)
    .round(3)
)

agg

## Step 3: Assessment of Missingness

The project focuses on missingness in `golddiffat15`, which was missing in about
**15.1%** of team rows.

The submitted reasoning argues that this missingness is **MAR rather than NMAR**:
whether a 15-minute gold-difference value is available is tied to the league/data
source and `datacompleteness`, not to the unobserved magnitude of the gold lead itself.

Two permutation tests compare the missingness indicator against:

- `league` — expected to show dependence;
- `side` — expected not to show dependence.

In [ ]:
# --- Missingness-dependency permutation tests for `golddiffat15` ---

def tvd(dist_a, dist_b):
    """Total Variation Distance between two aligned probability distributions."""
    return np.abs(dist_a - dist_b).sum() / 2

missing = team['golddiffat15'].isna().values

def tvd_perm_test(col, n_iter=5000, seed=0):
    """Permutation test: does `missing` depend on categorical `col`? Statistic = TVD."""
    np.random.seed(seed)

    cats = team[col].astype(str)
    levels = cats.unique()
    arr = cats.values

    def dist(mask):
        return (
            pd.Series(arr[mask])
            .value_counts(normalize=True)
            .reindex(levels, fill_value=0)
        )

    obs = tvd(dist(missing), dist(~missing))
    stats = np.empty(n_iter)

    for i in range(n_iter):
        s = np.random.permutation(missing)
        stats[i] = tvd(dist(s), dist(~s))

    return obs, np.mean(stats >= obs), stats

obs_league, p_league, null_league = tvd_perm_test('league', seed=101)
print(
    f"golddiffat15 missingness vs LEAGUE: "
    f"observed TVD = {obs_league:.4f}, p = {p_league:.4f}"
)

obs_side, p_side, _ = tvd_perm_test('side', seed=202)
print(
    f"golddiffat15 missingness vs SIDE: "
    f"observed TVD = {obs_side:.4f}, p = {p_side:.4f}"
)

fig = px.histogram(
    x=null_league,
    nbins=50,
    title='Step 3: Permutation null of TVD (golddiffat15 missingness vs league)',
    labels={'x': 'TVD between league distributions under H0'},
)
fig.add_vline(
    x=obs_league,
    line_dash='dash',
    line_color='red',
    annotation_text=f'observed = {obs_league:.3f}',
    annotation_position='top left',
)
fig.update_layout(template='plotly_white', height=400, width=750, showlegend=False)
fig.show()

The submitted run reported:

- Missingness vs `league`: **TVD ≈ 0.9926**, **p ≈ 0.0000**
- Missingness vs `side`: **TVD = 0.0000**, **p = 1.0000**

This supports the conclusion that missingness depends strongly on league but not on
map side.

## Step 4: Hypothesis Testing

### Research question
Is there a statistically significant difference in win rate between Blue-side and
Red-side teams?

- **Null hypothesis:** Blue and Red teams have the same win probability; the observed
  difference is due to chance.
- **Alternative hypothesis:** Blue and Red teams have different win probabilities.
- **Test statistic:** absolute difference in win rates,
  `|p̂_Blue - p̂_Red|`.

In [ ]:
# Observed test statistic

p_blue = team.loc[team['side'] == 'Blue', 'result'].mean()
p_red = team.loc[team['side'] == 'Red', 'result'].mean()
T_obs = abs(p_blue - p_red)

print(f'Observed Blue win rate: {p_blue:.4f}')
print(f'Observed Red win rate: {p_red:.4f}')
print(f'Observed |diff| (T_obs): {T_obs:.4f}')

In [ ]:
# --- Permutation test for the side effect ---

np.random.seed(42)

results = team['result'].values.astype(float)
n_blue = int((team['side'] == 'Blue').sum())

N = 10_000
perm_T = np.empty(N)

for i in range(N):
    shuffled = np.random.permutation(results)
    perm_T[i] = abs(
        shuffled[:n_blue].mean() -
        shuffled[n_blue:].mean()
    )

p_value = np.mean(perm_T >= T_obs)

print(f"Observed T_obs: {T_obs:.4f}")
print(f"Permutation p-value (N={N:,}): {p_value:.4f}")

fig = px.histogram(
    x=perm_T,
    nbins=50,
    title='Step 4: Permutation null of |p_Blue - p_Red| (side shuffled)',
    labels={'x': '|p_Blue - p_Red| under H0'},
)
fig.add_vline(
    x=T_obs,
    line_dash='dash',
    line_color='red',
    annotation_text=f'observed = {T_obs:.4f}',
    annotation_position='top right',
)
fig.update_layout(template='plotly_white', height=400, width=750, showlegend=False)
fig.show()

The submitted analysis reported:

- Blue win rate: **0.5248**
- Red win rate: **0.4752**
- Observed absolute difference: **0.0496**
- Permutation p-value with 10,000 permutations: **< 0.0001**

Therefore, the project rejects the null hypothesis at `alpha = 0.05`.

## Step 5: Framing a Prediction Problem

The target is `result`, making this a **binary classification** problem at the
team-game level.

Because each match has exactly one winner and one loser, the classes are balanced.
The project uses **accuracy** as the primary metric and **F1** as a secondary metric.

Only information available **before the match starts** is permitted in the predictive
models. Post-game features such as kills, gold, and `golddiffat15` are excluded to
avoid outcome leakage.

## Step 6: Baseline Model

The baseline uses two nominal pre-game features:

- `side`
- `league`

Both are one-hot encoded and passed to `LogisticRegression` in a scikit-learn
`Pipeline`. A 20% held-out test split is created once and reused later.

In [ ]:
# --- Baseline model (side + league -> result) ---

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

y = team['result'].astype(int)

train_idx, test_idx = train_test_split(
    team.index,
    test_size=0.2,
    random_state=42,
)

X_base = team[['side', 'league']]

pre_base = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['side', 'league']),
])

baseline = Pipeline([
    ('pre', pre_base),
    ('clf', LogisticRegression(max_iter=1000)),
])

baseline.fit(X_base.loc[train_idx], y.loc[train_idx])

base_test_pred = baseline.predict(X_base.loc[test_idx])
baseline_acc = accuracy_score(y.loc[test_idx], base_test_pred)
baseline_f1 = f1_score(y.loc[test_idx], base_test_pred)

n_ohe = (
    baseline.named_steps['pre']
    .transformers_[0][1]
    .get_feature_names_out()
    .shape[0]
)

print(
    'Baseline features: 2 nominal (side, league); '
    f'0 quantitative; 0 ordinal -> {n_ohe} one-hot columns'
)
print(f"Baseline TEST accuracy: {baseline_acc:.4f}")
print(f"Baseline TEST F1: {baseline_f1:.4f}")
print(f"Base rate (always-win): {y.loc[test_idx].mean():.4f}")

The submitted run reported:

- **Baseline accuracy:** 0.5104
- **Baseline F1:** 0.5127
- **Test-set base rate:** 0.5002

## Step 7: Final Model

The final model adds two leakage-free engineered features representing prior team form:

1. `team_prior_winrate`: each team's win rate over earlier games only, using
   `shift(1).expanding().mean()`.
2. `team_games_played`: number of prior games for the team.

It also adds `patch` and `playoffs`.

The model remains logistic regression, with `GridSearchCV` tuning
`clf__C` over `[0.01, 0.1, 1, 10, 100]` using 5-fold cross-validation on the
training set.

In [ ]:
# --- Final model with leakage-free engineered features ---

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV

tm = team.sort_values(['date', 'gameid']).copy()
grp = tm.groupby('teamname')['result']

tm['team_prior_winrate'] = grp.transform(
    lambda s: s.shift(1).expanding().mean()
)
tm['team_games_played'] = grp.cumcount()

tm = tm.sort_index()

feat_cols = [
    'side',
    'league',
    'patch',
    'playoffs',
    'team_prior_winrate',
    'team_games_played',
]

feats = tm[feat_cols].copy()
feats['playoffs'] = feats['playoffs'].astype(int)

print(
    'First-game rows with no prior history (imputed to 0.5):',
    int(feats['team_prior_winrate'].isna().sum()),
)

num_feats = ['team_prior_winrate', 'team_games_played']
cat_feats = ['side', 'league', 'patch']

pre_final = ColumnTransformer([
    (
        'num',
        Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value=0.5)),
            ('scale', StandardScaler()),
        ]),
        num_feats,
    ),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_feats),
    ('pass', 'passthrough', ['playoffs']),
])

final_pl = Pipeline([
    ('pre', pre_final),
    ('clf', LogisticRegression(max_iter=2000)),
])

grid = GridSearchCV(
    final_pl,
    {'clf__C': [0.01, 0.1, 1, 10, 100]},
    scoring='accuracy',
    cv=5,
)

grid.fit(feats.loc[train_idx], y.loc[train_idx])

best = grid.best_estimator_
final_test_pred = best.predict(feats.loc[test_idx])

final_test_acc = accuracy_score(y.loc[test_idx], final_test_pred)
final_test_f1 = f1_score(y.loc[test_idx], final_test_pred)
final_train_acc = accuracy_score(
    y.loc[train_idx],
    best.predict(feats.loc[train_idx]),
)

print(f"Best hyperparameter: {grid.best_params_}")
print(
    f"Final model TEST accuracy: {final_test_acc:.4f} "
    f"(F1 = {final_test_f1:.4f})"
)
print(f"Final model TRAIN accuracy: {final_train_acc:.4f}")
print(f"Baseline TEST accuracy: {baseline_acc:.4f}")
print(f"Improvement over baseline: {final_test_acc - baseline_acc:+.4f}")

best.fit(feats, y)
print("Final model refit on the full dataset for Step 8.")

The submitted run reported:

- First-game rows with no prior history: **654**
- Best hyperparameter: **`C = 0.01`**
- Final test accuracy: **0.5874**
- Final test F1: **0.6000**
- Final train accuracy: **0.5881**
- Improvement over baseline: **+0.0770**

## Step 8: Fairness Analysis

The final model is audited for performance differences between:

- **major:** LCK, LPL, LEC, LCS
- **other:** all remaining leagues

The evaluation metric is **precision**, with recall also reported. A permutation test
uses the absolute difference in precision between the groups as its statistic.

In [ ]:
# --- Fairness analysis: precision parity across league tiers ---

from sklearn.metrics import precision_score, recall_score

np.random.seed(7)

TIER1_MAJOR = ['LCK', 'LPL', 'LEC', 'LCS']
group = np.where(team['league'].isin(TIER1_MAJOR), 'major', 'other')

pred_all = best.predict(feats)

fair = pd.DataFrame({
    'y': y.values,
    'pred': pred_all,
    'group': group,
})

print('Per-group performance of the final model:')

for g in ['major', 'other']:
    sub = fair[fair['group'] == g]
    print(
        f" {g:6s} n={len(sub):6d} "
        f"precision={precision_score(sub['y'], sub['pred'], zero_division=0):.4f} "
        f"recall={recall_score(sub['y'], sub['pred'], zero_division=0):.4f}"
    )

def group_precision(sub):
    return precision_score(sub['y'], sub['pred'], zero_division=0)

obs_diff = abs(
    group_precision(fair[fair['group'] == 'major']) -
    group_precision(fair[fair['group'] == 'other'])
)

yv = fair['y'].values
pv = fair['pred'].values
gv = fair['group'].values

N = 5000
perm_stats = np.empty(N)

for i in range(N):
    sg = np.random.permutation(gv)
    is_major = sg == 'major'
    perm_stats[i] = abs(
        precision_score(yv[is_major], pv[is_major], zero_division=0) -
        precision_score(yv[~is_major], pv[~is_major], zero_division=0)
    )

p_fair = np.mean(perm_stats >= obs_diff)

print(f"Observed |precision gap|: {obs_diff:.4f}")
print(f"Permutation p-value: {p_fair:.4f}")

fig = px.histogram(
    x=perm_stats,
    nbins=50,
    title='Step 8: Permutation null of the precision gap (major vs other)',
    labels={'x': '|precision_major - precision_other| under H0'},
)

fig.add_vline(
    x=obs_diff,
    line_dash='dash',
    line_color='red',
    annotation_text=f'observed = {obs_diff:.4f}',
    annotation_position='top right',
)

fig.update_layout(
    template='plotly_white',
    height=400,
    width=750,
    showlegend=False,
)

fig.show()

The submitted run reported:

| Group | n | Precision | Recall |
|---|---:|---:|---:|
| major | 3,604 | 0.5991 | 0.6104 |
| other | 21,454 | 0.5797 | 0.6081 |

- Observed precision gap: **0.0194**
- Permutation p-value: **0.1168**

At `alpha = 0.05`, the submitted project **fails to reject the null hypothesis**:
there is no statistically significant evidence of a precision disparity between the
major and other league groups.